# Phase 3 — dual-T4 encoder sweep (Kaggle)
## Setup — do this first
## Expected wall-clock
## The grid (first slice — 3 cells × 12 seeds = 36 encoders)

## 1. Verify the GPUs (expect two Tesla T4s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download data + build image caches

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

## 4b. (Optional fast path) Reuse encoders / checkpoints you already trained

In [ ]:
import re, shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")




DEFAULT_CELL = "color_strong"
_FULL = re.compile(r"(?:color|position|control|orientation|scale)_(?:strong|weak)_seed\d+")

def _run_id(p):
    m = _FULL.search(p.as_posix())
    if m:
        return m.group(0)
    m = re.search(r"seed(\d+)", p.name)
    return f"{DEFAULT_CELL}_seed{m.group(1)}" if m else None

def _target(name):
    n = name.lower()
    if "ckpt" in n:
        return "last_ckpt.pt"
    if "backbone" in n:
        return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    tgt, rid = _target(p.name), _run_id(p)
    if tgt and rid:
        found.setdefault((rid, tgt), p)

if not found:
    print("no backbone*.pt / last_ckpt*.pt under /kaggle/input -- Add Input -> your dataset first")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"OK {rid}: {src.name} -> {dst.relative_to(REPO)} ({dst.stat().st_size >> 20} MB)")

## 5. Define the sweep

In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)



SEEDS = list(range(12))
CONFIGS = [
    ("configs/run/color_strong.yaml",    "color",    "strong"),
    ("configs/run/control_strong.yaml",  "control",  "strong"),
    ("configs/run/position_strong.yaml", "position", "strong"),
]
JOBS = [(cfg, cond, strg, s) for (cfg, cond, strg) in CONFIGS for s in SEEDS]

def run_id(cond, strg, seed): return f"{cond}_{strg}_seed{seed}"
def is_done(cond, strg, seed): return (ENC / run_id(cond, strg, seed) / "backbone.pt").exists()

print(f"{len(JOBS)} encoder cells: "
      + ", ".join(f"{c}_{s}x{len(SEEDS)}" for _, c, s in CONFIGS))

## 6. Run the sweep on both GPUs

In [ ]:
import os, sys, subprocess, time

N_GPUS = max(1, __import__("torch").cuda.device_count())

def _launch(job, gpu):
    cfg, cond, strg, seed = job
    rid = run_id(cond, strg, seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", cfg,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run_grid(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])
    time.sleep(2)
    queue = [j for j in JOBS if not is_done(j[1], j[2], j[3])]
    print(f"{len(JOBS)} cells | {len(JOBS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    job = queue.pop(0)
                    if is_done(job[1], job[2], job[3]):
                        continue
                    slots[g] = _launch(job, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    cond, strg, seed = next(j[1:] for j in JOBS if run_id(j[1], j[2], j[3]) == s["rid"])
                    ok = is_done(cond, strg, seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("sweep pass complete — all cells have a backbone.pt")

run_grid()

## 7. Monitor both GPUs

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 8. Progress summary

In [ ]:
from collections import Counter
import json as _json

EST_GPU_H_PER_CELL = 5.0

def _last_epoch(rid):
    log = ENC / rid / "train_log.jsonl"
    if not log.exists(): return None
    last = None
    for line in log.read_text().splitlines():
        try: last = _json.loads(line).get("epoch", last)
        except Exception: pass
    return last

rows = []
for cfg, cond, strg in CONFIGS:
    for s in SEEDS:
        rid = run_id(cond, strg, s); d = ENC / rid
        if (d / "backbone.pt").exists(): state = "done"
        elif (d / "last_ckpt.pt").exists(): state = "partial"
        else: state = "todo"
        rows.append((rid, state, _last_epoch(rid)))

c = Counter(st for _, st, _ in rows)
print(f"done={c['done']}  partial={c['partial']}  todo={c['todo']}  (of {len(rows)})\n")
for rid, state, ep in rows:
    tag = state + (f" @ep{ep}" if state == "partial" and ep is not None else "")
    print(f"  {rid:28s} {tag}")

remaining_cells = c["todo"] + 0.5 * c["partial"]
gpu_h = remaining_cells * EST_GPU_H_PER_CELL
n_gpus = max(1, __import__("torch").cuda.device_count())
print(f"\n~{gpu_h:.0f} GPU-h remaining -> ~{gpu_h / n_gpus:.0f} wall-clock h on {n_gpus} GPUs "
      f"(~{gpu_h / n_gpus / 30:.1f} weeks at 30 quota-h/week)")

## 9. Verify the 50-epoch cut on the first encoders

In [ ]:
import json as _json

for rid in ["color_strong_seed0", "position_strong_seed0"]:
    mp = ENC / rid / "metrics.json"
    if not mp.exists():
        print(f"{rid}: not trained yet"); continue
    m = _json.loads(mp.read_text()); d = m.get("diagnostics", {})
    print(f"{rid}: loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
          f"feat_std={d.get('feat_std'):.4f} eff_rank={d.get('eff_rank'):.1f} "
          f"align={d.get('alignment'):.3f} unif={d.get('uniformity'):.3f}")

print("\n--- Shapes3D quality gate on the first Color encoder ---")
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/color_strong.yaml \
    --simclr results/encoders/color_strong_seed0/backbone.pt \
    --random-seed 0 --out results/quality_gate/color_strong_seed0.json

## 10. Probe the trained encoders → `stacks.npz` (the interface contract)

In [ ]:
import subprocess, time, glob
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
N_SEEDS = 12
RANDOM_SEEDS = [str(i) for i in range(N_SEEDS)]


JOBS = [
    dict(dataset="shapes3d", condition="color",    strength="strong", extra=[]),
    dict(dataset="shapes3d", condition="control",  strength="strong", extra=[]),
    dict(dataset="dsprites", condition="position", strength="strong",
         extra=["--data-path", "data/raw/dsprites.npz"]),
]

results = {}
for job in JOBS:
    tag = f"{job['condition']}_{job['strength']}"
    print(f"\n{'='*72}\n[run-all] {tag}  ({job['dataset']})\n{'='*72}", flush=True)


    if (REPO / "results" / "probes" / tag / "stacks.npz").exists():
        print(f"[run-all] {tag}: stacks.npz already present -> skip.", flush=True)
        results[tag] = "skipped (done)"
        continue


    encs = sorted(glob.glob(str(REPO / f"results/encoders/{tag}_seed*/backbone.pt")))
    if len(encs) < N_SEEDS:
        print(f"[run-all] {tag}: only {len(encs)}/{N_SEEDS} encoders trained -> skip (train first).", flush=True)
        results[tag] = f"skipped ({len(encs)}/{N_SEEDS} encoders)"
        continue

    cmd = ["python", "-m", "src.probes.run_sweep",
           "--config", "configs/probe/ladder.yaml",
           "--dataset", job["dataset"],
           "--condition", job["condition"], "--strength", job["strength"],
           "--encoders", *encs,
           "--random-seed", *RANDOM_SEEDS,
           "--epochs", "100", "--num-workers", "2", *job["extra"]]
    t = time.time()
    rc = subprocess.run(cmd, cwd=str(REPO)).returncode
    dt = (time.time() - t) / 3600
    results[tag] = f"OK in {dt:.2f} h" if rc == 0 else f"FAILED (exit {rc}) after {dt:.2f} h"
    print(f"[run-all] {tag}: {results[tag]}", flush=True)

print(f"\n{'='*72}\n[run-all] SUMMARY\n{'='*72}")
for tag, st in results.items():
    print(f"  {tag:18s} {st}")


In [ ]:

import numpy as np, json as _json

for tag in ["color_strong", "control_strong", "position_strong"]:
    d = REPO / "results" / "probes" / tag
    if not (d / "stacks.npz").exists():
        print(f"{tag}: not produced yet"); continue
    z = np.load(d / "stacks.npz")
    meta = _json.loads((d / "meta.json").read_text())
    g = meta["quality_gate"]
    print(f"{tag}: arrays={ {k: z[k].shape for k in z.files} } | "
          f"factors={[f['name'] for f in meta['factors']]} | "
          f"gate {g['n_passed']}/{g['n_encoders']} passed")
